In [1]:
# Your main script (e.g., run.py or notebook cell)

from pathlib import Path
import torch.utils.data
import dataclasses
import numpy as np # For np.ndarray type hint

# From your project
# Ensure correct import paths for your project structure
from src.ssunet.datasets import TemporalSumSplitDataset, TemporalHalvesValidationDataset
from src.ssunet.models import Bit2Bit
from src.ssunet.configs import MasterConfig, SSUnetData
from src.ssunet.utils import LOGGER # Assuming LOGGER is set up

config_path = Path("config_ts.yml")

# --- Configuration Loading ---
config_loaded_successfully = False
try:
    config = MasterConfig.from_config(config_path)
    print(f"Successfully loaded configuration from: {config_path}")
    # Ensure target_path exists before copying
    if config.target_path: # Check if target_path is not None
        config.target_path.mkdir(parents=True, exist_ok=True)
        config.copy_config(config_path)
        print(f"Config copied to: {config.target_path / Path(config_path).name}")
    else:
        print("WARNING: config.target_path is not set, cannot copy config file.")
    config_loaded_successfully = True
except FileNotFoundError:
    print(f"ERROR: Configuration file not found at '{config_path}'.")
    # For a script, consider: import sys; sys.exit(1)
except Exception as e:
    print(f"ERROR loading configuration: {e}")
    # raise # Uncomment for full traceback during debugging

if config_loaded_successfully:
    # --- Data Loading ---
    print("Loading data...")
    data_loaded_successfully = False
    try:
        ssunet_training_data_source = config.path_config.load_data_only()
        if ssunet_training_data_source.primary_data.ndim not in [4, 5]:
            raise ValueError(f"Training data source expected as 4D (TDHW) or 5D (TCDHW), got {ssunet_training_data_source.primary_data.ndim}D")
        print(f"  Training data source loaded. Shape: {ssunet_training_data_source.primary_data.shape}")

        val_primary_source_cfg = config.path_config.reference
        if not val_primary_source_cfg or not val_primary_source_cfg.is_available:
             raise FileNotFoundError("Validation primary data (PATH.reference) not configured or available.")
        validation_primary_timeseries_array = config.path_config._load_from_source(val_primary_source_cfg, None)
        if validation_primary_timeseries_array is None:
             raise FileNotFoundError("Failed to load validation primary data (PATH.reference).")
        if validation_primary_timeseries_array.ndim not in [4, 5]:
            raise ValueError(f"Validation primary data expected as 4D (TDHW) or 5D (TCDHW), got {validation_primary_timeseries_array.ndim}D")
        print(f"  Validation primary (time-series) loaded. Shape: {validation_primary_timeseries_array.shape}")

        processed_spatial_gt_array = None # This will be 3D/4D or None
        if config.path_config.ground_truth and config.path_config.ground_truth.is_available:
            raw_validation_spatial_gt_array = config.path_config._load_from_source(config.path_config.ground_truth, None)
            
            if raw_validation_spatial_gt_array is not None:
                print(f"  Raw Validation spatial GT (for metrics) loaded. Shape: {raw_validation_spatial_gt_array.shape}")
                
                # --- FIX IS HERE: Ensure GT for metrics is 3D or 4D ---
                if raw_validation_spatial_gt_array.ndim == 5:
                    LOGGER.info("Raw spatial GT for metrics was 5D, taking first time slice (T=0) to make it 4D/3D.")
                    # Assuming T is the first dimension of the 5D array after loading
                    processed_spatial_gt_array = raw_validation_spatial_gt_array[0] 
                    print(f"    Sliced 5D GT to 4D/3D. New Shape for GT: {processed_spatial_gt_array.shape}")
                elif raw_validation_spatial_gt_array.ndim in [3, 4]:
                    processed_spatial_gt_array = raw_validation_spatial_gt_array
                else:
                    LOGGER.warning(f"Loaded spatial GT has unsupported ndim {raw_validation_spatial_gt_array.ndim}. "
                                   "It will not be used for metrics.")
                    processed_spatial_gt_array = None # Discard if not 3D/4D/5D
                
                if processed_spatial_gt_array is not None:
                    # Normalization should ideally be done by PathConfig's specific loading methods,
                    # but if _load_from_source is used directly, it returns raw data.
                    processed_spatial_gt_array = config.path_config._normalize_ground_truth(processed_spatial_gt_array)
                    if processed_spatial_gt_array is not None: # Normalization might return None if input was None
                         print(f"  Processed Validation spatial GT (for metrics). Shape: {processed_spatial_gt_array.shape}")
                    else:
                         print("  Validation spatial GT became None after normalization attempt.")
            else:
                print("  Configured ground_truth file could not be loaded.")
        else:
            print("  No ground_truth configured for validation metrics.")
        
        data_loaded_successfully = True
    except Exception as e:
        print(f"ERROR loading data: {e}")
        # raise 

    if data_loaded_successfully:
        print("Configuring and creating datasets...")
        data_config_main = config.data_config
        
        # Create a validation-specific config
        validation_specific_data_config = dataclasses.replace(data_config_main) # Create a mutable copy
        validation_specific_data_config.augments = False
        validation_specific_data_config.rotation = 0
        validation_specific_data_config.random_crop = True # Or False for consistent center patch
        if validation_specific_data_config.virtual_size == 0: # Ensure validation loader is not empty
            validation_specific_data_config.virtual_size = max(1, config.loader_config.batch_size * 5) # e.g. 5 batches
            print(f"  Validation virtual_size was 0, set to {validation_specific_data_config.virtual_size}")

        # Ensure training data is correctly handled (tensor conversion happens inside dataset)
        training_data = TemporalSumSplitDataset(
            ssunet_training_data_source, # SSUnetData with 5D/4D primary
            data_config_main,
            config.split_params
        )
        print(f"  Training dataset (TemporalSumSplitDataset) created. Length: {len(training_data)}")

        validation_data = TemporalHalvesValidationDataset(
            primary_timeseries_data=validation_primary_timeseries_array, 
            config=validation_specific_data_config,
            aggregation_halves="mean", 
            spatial_ground_truth_for_metrics=processed_spatial_gt_array # Pass the 3D/4D or None GT
        )
        print(f"  Validation dataset (TemporalHalvesValidationDataset) created. Length: {len(validation_data)}")

        # --- Model Creation ---
        print("Initializing model...")
        model = Bit2Bit(config.model_config)

        # --- Data Loaders ---
        print("Creating data loaders...")
        training_loader = config.loader_config.loader(training_data)
        print(f"  Training loader created. Shuffle: {getattr(training_loader, 'shuffle', 'N/A')}")

        val_loader_config_dict = dataclasses.asdict(config.loader_config)
        val_loader_config_dict['shuffle'] = False # Explicitly set for validation
        val_loader_config_dict['drop_last'] = False # Explicitly set for validation
        
        if len(validation_data) > 0 and len(validation_data) < val_loader_config_dict['batch_size'] and val_loader_config_dict['drop_last']:
            # This condition will now likely be false because drop_last is set to False above.
            # However, the check for dataset size vs batch size is still useful.
            print(f"INFO: Validation dataset size ({len(validation_data)}) is less than batch size "
                  f"({val_loader_config_dict['batch_size']}).")
        
        validation_loader = torch.utils.data.DataLoader(validation_data, **val_loader_config_dict)
        
        # --- CORRECTED PRINT STATEMENT ---
        print(f"  Validation loader created. Configured Shuffle: {val_loader_config_dict['shuffle']}, Configured Drop Last: {val_loader_config_dict['drop_last']}")
        # You can also check the sampler type for shuffle:
        if isinstance(validation_loader.sampler, torch.utils.data.RandomSampler):
            print("    Actual Sampler: RandomSampler (implies shuffle=True behavior)")
        elif isinstance(validation_loader.sampler, torch.utils.data.SequentialSampler):
            print("    Actual Sampler: SequentialSampler (implies shuffle=False behavior)")
        else:
            print(f"    Actual Sampler Type: {type(validation_loader.sampler)}")

        # --- Check Input Size (Optional Sanity Check) ---
        try:
            if len(training_loader) > 0:
                train_batch = next(iter(training_loader))
                model_input_train, target_for_loss_train = train_batch[0], train_batch[1]
                print(f"  TRAIN sample input batch shape: {tuple(model_input_train.shape)}")
                print(f"  TRAIN sample target batch shape: {tuple(target_for_loss_train.shape)}")
            else:
                print("WARNING: Training loader is empty.")

            if len(validation_loader) > 0: 
                val_batch = next(iter(validation_loader))
                model_input_val, target_for_loss_val = val_batch[0], val_batch[1]
                print(f"  VALIDATION sample input batch shape: {tuple(model_input_val.shape)}")
                print(f"  VALIDATION sample target batch shape: {tuple(target_for_loss_val.shape)}")
                if len(val_batch) > 2 and val_batch[2] is not None:
                    print(f"  VALIDATION sample true spatial GT batch shape: {tuple(val_batch[2].shape)}")
            else:
                print("WARNING: Validation loader is empty after adjustments.")
        except StopIteration:
            print("ERROR: A loader is empty. Check dataset sizes, batch_size and virtual_size.")
        except Exception as e:
            print(f"Error getting sample batch: {e}")
            # raise

        # --- Model Training ---
        print("Starting model training...")
        trainer = config.trainer
        try:
            trainer.fit(model, training_loader, validation_loader)
            print("Training completed.")
        except Exception as e:
            print(f"An error occurred during training: {e}")
            # raise
    else:
        print("Data loading failed. Skipping further steps.")
else:
    print("Configuration loading failed. Script cannot proceed.")

print("Script finished.")

Successfully loaded configuration from: config_ts.yml
Config copied to: models\20250509_172615_e=120_p=32_d=[0]_temporal_sum_split_on_TZYX_file_500x34x128x128_skip=1_l=10_d=5_sf=32_ds=2at10_f=10.0_z=3_g=8_sd=0_b=tri_a=gelu\config_ts.yml
Loading data...
  Training data source loaded. Shape: (31, 1, 34, 384, 1024)
  Validation primary (time-series) loaded. Shape: (31, 1, 34, 384, 1024)
  Raw Validation spatial GT (for metrics) loaded. Shape: (31, 1, 34, 384, 1024)
    Sliced 5D GT to 4D/3D. New Shape for GT: (1, 34, 384, 1024)
  Processed Validation spatial GT (for metrics). Shape: (1, 34, 384, 1024)
Configuring and creating datasets...
  Training dataset (TemporalSumSplitDataset) created. Length: 500
  Validation dataset (TemporalHalvesValidationDataset) created. Length: 500
Initializing model...
Creating data loaders...
  Training loader created. Shuffle: N/A
  Validation loader created. Configured Shuffle: False, Configured Drop Last: False
    Actual Sampler: SequentialSampler (impli

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


  VALIDATION sample input batch shape: (2, 1, 34, 128, 128)
  VALIDATION sample target batch shape: (2, 1, 34, 128, 128)
  VALIDATION sample true spatial GT batch shape: (2, 1, 34, 128, 128)
Starting model training...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type                             | Params | Mode 
-------------------------------------------------------------------------
0 | psnr_metric | PeakSignalNoiseRatio             | 0      | train
1 | ssim_metric | StructuralSimilarityIndexMeasure | 0      | train
2 | down_convs  | ModuleList                       | 9.6 M  | train
3 | up_convs    | ModuleList                       | 5.1 M  | train
4 | conv_final  | Sequential                       | 33     | train
-------------------------------------------------------------------------
14.7 M    Trainable params
0         Non-trainable params
14.7 M    Total params
58.718    Total estimated model params size (MB)
100       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

ShapeMismatchError: Data and reference shapes do not match


An error occurred during training: Data and reference shapes do not match
Script finished.
